# Rank Diagnosis Using Medication

## Architectural Flow

![Architectural Flow](architecture_flow.png)

# Rank Diagnosis Using Medication 

This notebook ranks a list of candidate diagnosis provided with a list of treatments (medication). Using the IMO Health Knowledge Graph  medication edges to Problem concepts as the basis of clinical evidence to give condidence scores to the candidate diagnosis

## Actual tool chain used in `rank` mode
```python
RANK_TOOLS = [
    normalize_medication_only,   # Step 1
    get_lexical_for_ranking,     # Step 2  (always calls _code_based_filter)
]
```

## Key implementation details matched here
- `_code_based_filter` always runs for every medication (no list-size threshold)
- `_code_based_filter` normalizes candidates to IMO codes and checks **direct code match + broader parent codes** against the KG list
- `MedicationLexical` GraphQL query returns `treatedProblems`, `preventedProblems`, `causedProblems` only

## Scoring Formula (baseline: **0.50**) — from RANK_SYSTEM_PROMPT

| Signal | Delta |
|---|---|
| Direct `treatedProblems` match (title or synonym) | **+0.20** |
| Hierarchy match (candidate is a KG child of a treatedProblem) | **+0.15** |
| Each additional medication supporting same candidate | **+0.10** |
| No KG evidence | 0 (stays 0.50 Neutral) |

**Score labels:** 0.80–1.00 Strong · 0.60–0.79 Moderate · 0.40–0.59 Neutral · 0.20–0.39 Weak · 0.00–0.19 Contraindicated

## 0. Setup — paths, imports, credentials

In [ ]:
import sys
import os

# Ensure the notebook's own directory is on the path so local
# kg_config.py and kg_api_client.py are imported directly.
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

import json
from IPython.display import display, Markdown

# ── Credential check ────────────────────────────────────────────────────────
# kg_config.py reads credentials from AWS SSM or environment variables.
# If SSM is unavailable locally, uncomment and set the env vars below.
#
# os.environ["IMO_NORMALIZE_CLIENT_ID"] = "..."
# os.environ["IMO_NORMALIZE_SECRET"]    = "..."
# os.environ["IMO_KG_CLIENT_ID"]        = "..."
# os.environ["IMO_KG_CLIENT_SECRET"]    = "..."

import kg_config as kc

missing = []
if not kc.IMO_NORMALIZE_CLIENT_ID:     missing.append("IMO_NORMALIZE_CLIENT_ID")
if not kc.IMO_NORMALIZE_CLIENT_SECRET: missing.append("IMO_NORMALIZE_SECRET")
if not kc.IMO_KG_CLIENT_ID:           missing.append("IMO_KG_CLIENT_ID")
if not kc.IMO_KG_CLIENT_SECRET:       missing.append("IMO_KG_CLIENT_SECRET")

if missing:
    print(f"⚠  Missing credentials: {missing}")
    print("   Set them as env vars (see comment above) or ensure AWS SSM access.")
else:
    print("✓  Credentials loaded")
    print(f"   Normalize URL : {kc.IMO_NORMALIZE_URL}")
    print(f"   KG GraphQL URL: {kc.KG_GRAPHQL_URL}")

✓  Credentials loaded
   Normalize URL : https://api.imohealth.com/precision/normalize
   KG GraphQL URL: https://api.imohealth.com/knowledgegraph/graphql/


## 1. Clinical Scenario

A 68-year-old male presents with palpitations, fatigue, and reduced exercise tolerance.
ECG shows irregular rhythm with a prior episode of sustained tachycardia.
Echo confirms reduced ejection fraction (~35%) with global LV dilation.

**Candidate diagnoses to rank:**
1. Paroxysmal atrial fibrillation
2. Ventricular tachycardia
3. Hypertrophic cardiomyopathy
4. Chronic atrial fibrillation
5. Pulmonary embolism

**Current medications** (the evidence we rank against):
- Digoxin 0.125 mg daily
- Warfarin 5 mg daily
- Metoprolol 25 mg BID

In [1]:
CANDIDATE_DIAGNOSES = [
    "Paroxysmal atrial fibrillation",
    "Ventricular tachycardia",
    "Hypertrophic cardiomyopathy",
    "Chronic atrial fibrillation",
    "Pulmonary embolism",
]

MEDICATIONS = [
    "Digoxin 0.125 mg daily",
    "Warfarin 5 mg daily",
    "Metoprolol 25 mg BID",
]

print(f"Candidates : {CANDIDATE_DIAGNOSES}")
print(f"Medications: {MEDICATIONS}")

Candidates : ['Paroxysmal atrial fibrillation', 'Ventricular tachycardia', 'Hypertrophic cardiomyopathy', 'Chronic atrial fibrillation', 'Pulmonary embolism']
Medications: ['Digoxin 0.125 mg daily', 'Warfarin 5 mg daily', 'Metoprolol 25 mg BID']


## 2. Initialize KGApiClient

In [2]:
from kg_api_client import KGApiClient

client = KGApiClient()
print("KGApiClient ready")

KGApiClient ready


## Step 1 — Normalize each medication (`normalize_medication_only`)

Calls `normalize_medical_term(input_term, domain="Medication")` for each medication.
Returns a `default_lexical_code` — the stable IMO identifier used for all KG lookups.

**Rule:** Only medications are normalized here. Candidate diagnoses are normalized in Step 2b
inside `_code_based_filter`, which now runs unconditionally for every medication.

In [3]:
import re

def strip_dosage(med_name: str) -> str:
    """Remove dose/frequency/route — the LLM extraction step outputs bare drug names,
    dosages are never passed to normalize_medication_only.
    Examples:
      'Digoxin 0.125 mg daily'  → 'Digoxin'
      'Metoprolol 25 mg BID'    → 'Metoprolol'
      'Warfarin 5 mg daily'     → 'Warfarin'
    """
    cleaned = re.split(r'\s+\d|\s+(mg|mcg|mEq|units?|tab|cap|BID|TID|QID|daily|once|twice|weekly)\b',
                       med_name, flags=re.IGNORECASE)[0].strip()
    return cleaned


med_codes = {}  # medication name -> {code, title, score, stripped_name}

print("Normalizing medications (domain=Medication)...\n")
for med in MEDICATIONS:
    stripped = strip_dosage(med)
    result = client.normalize_medical_term(stripped, "Medication")
    if result.get("success") and result.get("results"):
        matches = result["results"][0].get("matches", [])
        if matches:
            best = matches[0]
            med_codes[med] = {
                "code":          best["default_lexical_code"],
                "title":         best["title"],
                "score":         best["score"],
                "stripped_name": stripped,
            }
            suffix = f"  (stripped: '{stripped}')" if stripped != med else ""
            print(f"  ✓  {med:<32}  code={best['default_lexical_code']:<10}  title='{best['title']}'{suffix}")
        else:
            print(f"  ✗  {med:<32}  no matches returned (stripped: '{stripped}')")
    else:
        print(f"  ✗  {med:<32}  API error: {result.get('error')}")

print(f"\n✓  Normalized {len(med_codes)}/{len(MEDICATIONS)} medications")

Normalizing medications (domain=Medication)...

  ✓  Digoxin 0.125 mg daily            code=113675      title='digoxin'  (stripped: 'Digoxin')
  ✓  Warfarin 5 mg daily               code=129822      title='warfarin'  (stripped: 'Warfarin')
  ✓  Metoprolol 25 mg BID              code=129120      title='metoprolol'  (stripped: 'Metoprolol')

✓  Normalized 3/3 medications


## Step 2 — Query KG for each medication (`get_lexical_for_ranking`)

Calls `get_lexical(code, domain="medication")` against the IMO HEALTH KG GraphQL API,
then always runs `_code_based_filter` on the result:

```python
def get_lexical_for_ranking(imo_lexical_code, domain="Problem", candidate_diagnoses=""):

    result = _kg_client.get_lexical(imo_lexical_code, domain)

    # Only parse candidates if the argument was passed
    candidates = [c.strip() for c in candidate_diagnoses.split(",") if c.strip()] if candidate_diagnoses else None

    # _code_based_filter is always called when candidates are present —
    # runs for every medication regardless of KG list size
    if candidates and result.get("success") and result.get("lexical"):
        result = _code_based_filter(result, candidates)

    return _cap_tool_response(result)
```

The `MedicationLexical` GraphQL query returns:
- `treatedProblems { code title }`
- `preventedProblems { code title }`
- `causedProblems { code title }`

In [4]:
# Raw KG data per medication — mirrors _kg_client.get_lexical() inside get_lexical_for_ranking
kg_raw = {}  # medication name -> raw lexical result

print("Calling get_lexical(domain='medication') for each medication...\n")
for med, info in med_codes.items():
    code = info["code"]
    result = client.get_lexical(code, domain="medication")
    if result.get("success") and result.get("lexical"):
        lex = result["lexical"]
        treated   = lex.get("treatedProblems", []) or []
        prevented = lex.get("preventedProblems", []) or []
        caused    = lex.get("causedProblems", []) or []
        kg_raw[med] = {
            "code":              code,
            "lexical":           lex,
            "treatedProblems":   treated,
            "preventedProblems": prevented,
            "causedProblems":    caused,
        }
        print(
            f"  ✓  {med:<32}  "
            f"treatedProblems={len(treated):>4}  "
            f"preventedProblems={len(prevented):>4}  "
            f"causedProblems={len(caused):>4}  "
            f"→ _code_based_filter will run"
        )
    elif result.get("success") and not result.get("lexical"):
        print(f"  ✗  {med:<32}  KG returned null lexical for code={code} — code may be invalid or not a MedicationLexical")
    else:
        print(f"  ✗  {med:<32}  API error: {result.get('error', 'unknown')}")

print(f"\n✓  KG data fetched for {len(kg_raw)}/{len(med_codes)} medications")

Calling get_lexical(domain='medication') for each medication...

  ✓  Digoxin 0.125 mg daily            treatedProblems=4312  preventedProblems=   0  causedProblems=  11  → _code_based_filter will run
  ✓  Warfarin 5 mg daily               treatedProblems=30209  preventedProblems=   0  causedProblems=  24  → _code_based_filter will run
  ✓  Metoprolol 25 mg BID              treatedProblems=14643  preventedProblems=   0  causedProblems=   3  → _code_based_filter will run

✓  KG data fetched for 3/3 medications


### Step 2b — `_code_based_filter` (always runs)

**Fires for every medication regardless of `len(treatedProblems)`.**
For each medication:
1. Normalizes each candidate diagnosis (`domain=Problem`) to its IMO code
2. Fetches each candidate's `domainBroader` (parent) codes
3. Does **exact code matching** — candidate's own code or any parent code present in the KG list

Candidates that fail to normalize are recorded as unmatched and stay Neutral (0.50).

In [5]:
# Per-medication filtered treatedProblems + how each was resolved
# Structure: kg_data[med] = {
#   "treatedProblems": [...],          # items returned after _code_based_filter
#   "filter_applied": True,
#   "candidate_codes": {...}
# }
kg_data = {}

# Candidate IMO codes — normalized once and reused across all medications
candidate_info = {}  # candidate text -> {code, title, broader_codes}

def normalize_candidates_for_filter(candidates):
    """Normalize each candidate diagnosis to its IMO lexical code and fetch parent codes."""
    result = {}
    for cand in candidates:
        norm = client.normalize_medical_term(cand, "Problem")
        if norm.get("success") and norm.get("results"):
            matches = norm["results"][0].get("matches", [])
            if matches:
                code  = matches[0].get("default_lexical_code", "")
                title = matches[0].get("title", "")
                if code:
                    # Fetch broader (parent) codes
                    broader_codes = set()
                    hier = client.get_domain_hierarchy(code, "broader", "Problem")
                    if hier.get("success") and hier.get("lexical"):
                        for parent in (hier["lexical"].get("domainBroader") or []):
                            if parent.get("code"):
                                broader_codes.add(parent["code"])
                    result[cand] = {"code": code, "title": title, "broader_codes": broader_codes}
    return result


def code_based_filter(treated_list, candidate_codes_map):
    """Exact code matching: direct code match, then parent (broader) code match."""
    kg_codes = {item.get("code"): item for item in treated_list if item.get("code")}
    matched = []
    matched_codes = set()

    for cand_text, cand_info in candidate_codes_map.items():
        # Direct code match
        if cand_info["code"] in kg_codes:
            matched.append({**kg_codes[cand_info["code"]], "_match_type": "code-direct", "_candidate": cand_text})
            matched_codes.add(cand_info["code"])
        # Broader/parent code match
        for b_code in cand_info["broader_codes"]:
            if b_code in kg_codes and b_code not in matched_codes:
                matched.append({**kg_codes[b_code], "_match_type": "code-hierarchy", "_candidate": cand_text})
                matched_codes.add(b_code)

    return matched


print("Applying _code_based_filter to every medication...\n")

# Normalize candidates once — reused for every medication
print("  Normalizing candidate diagnoses (domain=Problem)...")
candidate_info.update(normalize_candidates_for_filter(CANDIDATE_DIAGNOSES))
for dx, info in candidate_info.items():
    print(f"    ✓  {dx:<35} code={info['code']}  parents={info['broader_codes']}")
print()

unmatched_candidates = [c for c in CANDIDATE_DIAGNOSES if c not in candidate_info]

for med, raw in kg_raw.items():
    treated = raw["treatedProblems"]
    matched = code_based_filter(treated, candidate_info)
    kg_data[med] = {
        "treatedProblems": matched,
        "filter_applied":  True,
        "total_in_kg":     len(treated),
        "unmatched_candidates": unmatched_candidates,
    }
    print(
        f"  ✓  {med:<28}  [_code_based_filter]  "
        f"KG total={len(treated):>5}  matched={len(matched)}  "
        f"unmatched={unmatched_candidates or '(none)'}"
    )
    for item in matched:
        print(
            f"       • '{item.get('title','')}' "
            f"(code={item.get('code','?')}, {item.get('_match_type','?')}) "
            f"← {item.get('_candidate','?')}"
        )

Applying _code_based_filter to every medication...

  Normalizing candidate diagnoses (domain=Problem)...
    ✓  Paroxysmal atrial fibrillation      code=72155  parents={'3944', '48464167'}
    ✓  Ventricular tachycardia             code=56802  parents={'56802', '45645', '55416', '3957', '385680'}
    ✓  Hypertrophic cardiomyopathy         code=3808  parents={'29243884', '87148'}
    ✓  Chronic atrial fibrillation         code=598308  parents={'3944', '43215879'}
    ✓  Pulmonary embolism                  code=76131  parents={'4032', '55237845'}

  ✓  Digoxin 0.125 mg daily        [_code_based_filter]  KG total= 4312  matched=8  unmatched=(none)
       • 'paroxysmal atrial fibrillation' (code=72155, code-direct) ← Paroxysmal atrial fibrillation
       • 'atrial fibrillation' (code=3944, code-hierarchy) ← Paroxysmal atrial fibrillation
       • 'ventricular tachycardia' (code=56802, code-direct) ← Ventricular tachycardia
       • 'tachycardia' (code=55416, code-hierarchy) ← Ventricular 

## Step 3 — Score each candidate

Applies the scoring formula from `RANK_SYSTEM_PROMPT`:

```
baseline = 0.50
+ 0.20  direct treatedProblems match          (first supporting med)
+ 0.15  hierarchy treatedProblems match        (first supporting med)
+ 0.10  each additional supporting medication
```

Unmatched candidates (in `_treatedProblems_unmatched_candidates`) stay at 0.50 (Neutral).

In [6]:
# Build match_evidence: dx -> list of {med, match_type}
# match_type: "code-direct" | "code-hierarchy"
match_evidence = {dx: [] for dx in CANDIDATE_DIAGNOSES}

for med, data in kg_data.items():
    for item in data["treatedProblems"]:
        cand = item.get("_candidate")
        mtype = item.get("_match_type", "unknown")
        if cand and cand in match_evidence:
            # Avoid duplicates (a candidate may appear multiple times in expanded list)
            if not any(e["med"] == med for e in match_evidence[cand]):
                match_evidence[cand].append({
                    "med":        med,
                    "match_type": mtype,
                    "kg_code":    item.get("code", ""),
                    "kg_title":   item.get("title", ""),
                })

# Also flag candidates the filter marked as unmatched
unmatched_by_filter = set()
for med, data in kg_data.items():
    for cand in data.get("unmatched_candidates", []):
        unmatched_by_filter.add(cand)

print("Match evidence summary:\n")
for dx in CANDIDATE_DIAGNOSES:
    evs = match_evidence[dx]
    if evs:
        for ev in evs:
            print(f"  {dx:<35}  ← {ev['med']:<28} [{ev['match_type']}]  KG: '{ev['kg_title']}' ({ev['kg_code']})")
    else:
        note = " (could not normalize — Neutral forced)" if dx in unmatched_by_filter else " (no KG evidence — Neutral)"
        print(f"  {dx:<35}  ← (none){note}")

Match evidence summary:

  Paroxysmal atrial fibrillation       ← Digoxin 0.125 mg daily       [code-direct]  KG: 'paroxysmal atrial fibrillation' (72155)
  Paroxysmal atrial fibrillation       ← Metoprolol 25 mg BID         [code-direct]  KG: 'paroxysmal atrial fibrillation' (72155)
  Ventricular tachycardia              ← Digoxin 0.125 mg daily       [code-direct]  KG: 'ventricular tachycardia' (56802)
  Ventricular tachycardia              ← Metoprolol 25 mg BID         [code-direct]  KG: 'ventricular tachycardia' (56802)
  Hypertrophic cardiomyopathy          ← Digoxin 0.125 mg daily       [code-direct]  KG: 'hypertrophic cardiomyopathy' (3808)
  Hypertrophic cardiomyopathy          ← Warfarin 5 mg daily          [code-direct]  KG: 'hypertrophic cardiomyopathy' (3808)
  Hypertrophic cardiomyopathy          ← Metoprolol 25 mg BID         [code-direct]  KG: 'hypertrophic cardiomyopathy' (3808)
  Chronic atrial fibrillation          ← Digoxin 0.125 mg daily       [code-direct]  KG: 'c

In [7]:
def score_label(score):
    if score >= 0.80: return "Strong"
    if score >= 0.60: return "Moderate"
    if score >= 0.40: return "Neutral"
    if score >= 0.20: return "Weak"
    return "Contraindicated"


scoring_details = {}

for dx in CANDIDATE_DIAGNOSES:
    evs = match_evidence[dx]
    score = 0.50
    breakdown = []
    first_hit = True

    if dx in unmatched_by_filter:
        # agent.py rule: "score them Neutral" — stays at 0.50
        scoring_details[dx] = {
            "score": 0.50, "label": "Neutral",
            "evidence": evs,
            "breakdown": ["Could not normalize candidate (unmatched by _code_based_filter) → Neutral 0.50"],
        }
        continue

    for ev in evs:
        mtype = ev["match_type"]
        if first_hit:
            # First medication: direct code match (+0.20) or hierarchy/other (+0.15)
            delta = 0.20 if mtype == "code-direct" else 0.15
            score += delta
            breakdown.append(f"+{delta:.2f}  {ev['med']}  [{mtype}]  KG: '{ev['kg_title']}'")
            first_hit = False
        else:
            # Each additional medication: +0.10
            score += 0.10
            breakdown.append(f"+0.10  {ev['med']}  [{mtype}]  (additional med)")

    score = max(0.0, min(1.0, score))
    scoring_details[dx] = {
        "score":     score,
        "label":     score_label(score),
        "evidence":  evs,
        "breakdown": breakdown,
    }

print("Score breakdown per candidate:\n")
for dx, det in scoring_details.items():
    print(f"{'─'*60}")
    print(f"  {dx}")
    print(f"  Baseline: 0.50")
    for line in det["breakdown"] or ["  (no KG evidence)"]:
        print(f"    {line}")
    print(f"  Final score: {det['score']:.2f}  →  {det['label']}")

Score breakdown per candidate:

────────────────────────────────────────────────────────────
  Paroxysmal atrial fibrillation
  Baseline: 0.50
    +0.20  Digoxin 0.125 mg daily  [code-direct]  KG: 'paroxysmal atrial fibrillation'
    +0.10  Metoprolol 25 mg BID  [code-direct]  (additional med)
  Final score: 0.80  →  Moderate
────────────────────────────────────────────────────────────
  Ventricular tachycardia
  Baseline: 0.50
    +0.20  Digoxin 0.125 mg daily  [code-direct]  KG: 'ventricular tachycardia'
    +0.10  Metoprolol 25 mg BID  [code-direct]  (additional med)
  Final score: 0.80  →  Moderate
────────────────────────────────────────────────────────────
  Hypertrophic cardiomyopathy
  Baseline: 0.50
    +0.20  Digoxin 0.125 mg daily  [code-direct]  KG: 'hypertrophic cardiomyopathy'
    +0.10  Warfarin 5 mg daily  [code-direct]  (additional med)
    +0.10  Metoprolol 25 mg BID  [code-direct]  (additional med)
  Final score: 0.90  →  Strong
──────────────────────────────────────

## Step 5 — Ranked Results



In [10]:
from IPython.display import display, Markdown
ranked = sorted(scoring_details.items(), key=lambda x: x[1]["score"], reverse=True)

lines = [
    "## Medication-Based Diagnosis Ranking",
    "",
    "| Rank | Diagnosis | Score | Label | Supporting Medications | Match Type |",
    "|------|-----------|-------|-------|------------------------|------------|",
]
for rank, (dx, det) in enumerate(ranked, 1):
    meds_str = ", ".join(
        f"{ev['med']} [{ev['match_type']}]" for ev in det["evidence"]
    ) or "—"
    match_types = ", ".join(sorted({ev['match_type'] for ev in det["evidence"]})) or "—"
    lines.append(f"| {rank} | {dx} | {det['score']:.2f} | {det['label']} | {meds_str} | {match_types} |")

display(Markdown("\n".join(lines)))

## Medication-Based Diagnosis Ranking

| Rank | Diagnosis | Score | Label | Supporting Medications | Match Type |
|------|-----------|-------|-------|------------------------|------------|
| 1 | Hypertrophic cardiomyopathy | 0.90 | Strong | Digoxin 0.125 mg daily [code-direct], Warfarin 5 mg daily [code-direct], Metoprolol 25 mg BID [code-direct] | code-direct |
| 2 | Paroxysmal atrial fibrillation | 0.80 | Moderate | Digoxin 0.125 mg daily [code-direct], Metoprolol 25 mg BID [code-direct] | code-direct |
| 3 | Ventricular tachycardia | 0.80 | Moderate | Digoxin 0.125 mg daily [code-direct], Metoprolol 25 mg BID [code-direct] | code-direct |
| 4 | Chronic atrial fibrillation | 0.80 | Moderate | Digoxin 0.125 mg daily [code-direct], Metoprolol 25 mg BID [code-direct] | code-direct |
| 5 | Pulmonary embolism | 0.70 | Moderate | Warfarin 5 mg daily [code-direct] | code-direct |

## Evidence Details

For each ranked candidate — exact KG match paths as the agent would include in its output.

In [11]:
for rank, (dx, det) in enumerate(ranked, 1):
    info = candidate_info.get(dx, {})
    print(f"{'─'*70}")
    print(f"  #{rank}  {dx}")
    print(f"       IMO Lexical Code : {info.get('code', 'N/A')}")
    print(f"       Normalized Title : {info.get('title', 'N/A')}")
    print(f"       Score / Label    : {det['score']:.2f} / {det['label']}")

    if det["evidence"]:
        print(f"       KG Evidence     :")
        for ev in det["evidence"]:
            print(f"         {ev['med']} → treatedProblems → '{ev['kg_title']}' (code={ev['kg_code']})  [{ev['match_type']}]")
    else:
        print(f"       KG Evidence     : (none — Neutral 0.50)")

    if info.get("broader_codes"):
        print(f"       Parent codes    : {info['broader_codes']}")
    print()

──────────────────────────────────────────────────────────────────────
  #1  Hypertrophic cardiomyopathy
       IMO Lexical Code : 3808
       Normalized Title : Hypertrophic cardiomyopathy
       Score / Label    : 0.90 / Strong
       KG Evidence     :
         Digoxin 0.125 mg daily → treatedProblems → 'hypertrophic cardiomyopathy' (code=3808)  [code-direct]
         Warfarin 5 mg daily → treatedProblems → 'hypertrophic cardiomyopathy' (code=3808)  [code-direct]
         Metoprolol 25 mg BID → treatedProblems → 'hypertrophic cardiomyopathy' (code=3808)  [code-direct]
       Parent codes    : {'29243884', '87148'}

──────────────────────────────────────────────────────────────────────
  #2  Paroxysmal atrial fibrillation
       IMO Lexical Code : 72155
       Normalized Title : Paroxysmal atrial fibrillation
       Score / Label    : 0.80 / Moderate
       KG Evidence     :
         Digoxin 0.125 mg daily → treatedProblems → 'paroxysmal atrial fibrillation' (code=72155)  [code-direct

## Appendix A — Raw KG Data per Medication

Shows the fields returned by the `MedicationLexical` GraphQL query: `treatedProblems`, `preventedProblems`, `causedProblems`.

In [12]:
PREVIEW_N = 15
candidate_codes_all = {info["code"] for info in candidate_info.values() if info.get("code")}

for med, raw in kg_raw.items():
    treated   = raw["treatedProblems"]
    prevented = raw["preventedProblems"]
    caused    = raw["causedProblems"]
    med_code  = raw["code"]

    print(f"{'='*70}")
    print(f"  {med}  (code: {med_code})")
    print(f"  KG Studio: https://studio.imohealth.com/#/terminology-browser/graph?id={med_code}")

    for field_name, field_data in [("treatedProblems", treated), ("preventedProblems", prevented), ("causedProblems", caused)]:
        if not field_data:
            print(f"\n  {field_name}: (empty)")
            continue
        print(f"\n  {field_name}  ({len(field_data)} total — showing first {min(PREVIEW_N, len(field_data))})")
        for item in field_data[:PREVIEW_N]:
            marker = "  ◀ CANDIDATE" if item.get("code") in candidate_codes_all else ""
            print(f"    code={item.get('code','?'):<10}  {item.get('title','')}{marker}")
        if len(field_data) > PREVIEW_N:
            print(f"    ... and {len(field_data) - PREVIEW_N} more")
    print()

  Digoxin 0.125 mg daily  (code: 113675)
  KG Studio: https://studio.imohealth.com/#/terminology-browser/graph?id=113675

  treatedProblems  (4312 total — showing first 15)
    code=964980      A-H prolongation
    code=29246685    abnormal aortic cusp
    code=1493760181  abnormal attachment of tricuspid chordae tendinea
    code=1493647377  abnormal course of aortic arch
    code=1493647566  abnormal course of aortic arch and descending aorta
    code=807935      abnormal heart rhythm due to congenital heart abnormality
    code=29243292    abnormal inferior vena caval connection
    code=1495715418  abnormal intrapericardial course of great arteries
    code=1494814073  abnormal lead impedance of cardiac resynchronization therapy defibrillator (CRT-D)
    code=1494814074  abnormal lead impedance of cardiac resynchronization therapy pacemaker (CRT-P)
    code=58053935    abnormal lead impedance of implantable cardioverter-defibrillator (ICD)
    code=1710908     abnormal left aortic 

## Appendix B — Consider Also

Conditions supported by **2+ medications** in `treatedProblems` but not in our candidate list.


In [13]:
from collections import defaultdict

condition_support = defaultdict(list)  # (code, title) -> [med names]

for med, raw in kg_raw.items():
    for item in raw["treatedProblems"]:
        code = item.get("code")
        if code and code not in candidate_codes_all:
            condition_support[(code, item.get("title", ""))].append(med)

consider_also = [
    (code, title, meds)
    for (code, title), meds in condition_support.items()
    if len(meds) >= 2
]
consider_also.sort(key=lambda x: len(x[2]), reverse=True)

if consider_also:
    lines = [
        "## Consider Also — conditions supported by 2+ medications (not in candidate list)",
        "",
        "| Condition | Code | Meds | Medications |",
        "|-----------|------|------|-------------|",
    ]
    for code, title, meds in consider_also[:10]:
        lines.append(f"| {title} | {code} | {len(meds)} | {', '.join(meds)} |")
    display(Markdown("\n".join(lines)))
else:
    print("No additional conditions found supported by 2+ medications.")

## Consider Also — conditions supported by 2+ medications (not in candidate list)

| Condition | Code | Meds | Medications |
|-----------|------|------|-------------|
| abnormal position of heart | 6009 | 3 | Digoxin 0.125 mg daily, Warfarin 5 mg daily, Metoprolol 25 mg BID |
| abnormal result of cardiovascular function study suggestive of non-ST elevation myocardial infarction (NSTEMI) | 59934189 | 3 | Digoxin 0.125 mg daily, Warfarin 5 mg daily, Metoprolol 25 mg BID |
| abnormal univentricular atrioventricular connection | 81518831 | 3 | Digoxin 0.125 mg daily, Warfarin 5 mg daily, Metoprolol 25 mg BID |
| abnormality of atrioventricular valve leaflet in atrioventricular septal defect | 29245508 | 3 | Digoxin 0.125 mg daily, Warfarin 5 mg daily, Metoprolol 25 mg BID |
| abnormality of common atrioventricular valve in atrioventricular septal defect | 29245235 | 3 | Digoxin 0.125 mg daily, Warfarin 5 mg daily, Metoprolol 25 mg BID |
| aborted myocardial infarction | 48670796 | 3 | Digoxin 0.125 mg daily, Warfarin 5 mg daily, Metoprolol 25 mg BID |
| absence of heart valve | 500624 | 3 | Digoxin 0.125 mg daily, Warfarin 5 mg daily, Metoprolol 25 mg BID |
| ACC/AHA stage B congestive heart failure due to ischemic cardiomyopathy | 91454105 | 3 | Digoxin 0.125 mg daily, Warfarin 5 mg daily, Metoprolol 25 mg BID |
| ACC/AHA stage B diastolic heart failure | 51211534 | 3 | Digoxin 0.125 mg daily, Warfarin 5 mg daily, Metoprolol 25 mg BID |
| ACC/AHA stage B systolic heart failure | 51211493 | 3 | Digoxin 0.125 mg daily, Warfarin 5 mg daily, Metoprolol 25 mg BID |